# 08 Business Clustering
Aggregate business features and cluster SMEs into strategic segments.

In [1]:
import warnings
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

warnings.filterwarnings("ignore")
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "utils" / "utils.py").exists():
    if (PROJECT_ROOT / "notebooks" / "utils" / "utils.py").exists():
        PROJECT_ROOT = PROJECT_ROOT / "notebooks"
    elif (PROJECT_ROOT.parent / "utils" / "utils.py").exists():
        PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from utils.utils import ensure_project_dirs, load_raw_dataset, clean_dataset, PROCESSED_DIR, REPORTS_DIR, FIGURES_DIR
from utils.features import engineer_kpis, build_post_feature_sets, aggregate_business_features
from utils.evaluation import regression_metrics, rank_models
from utils.visualization import set_plot_style, save_figure

set_plot_style()
ensure_project_dirs()
RAW_DATA_PATH = Path("../jsons/all_final_appended.json")
if not RAW_DATA_PATH.exists():
    RAW_DATA_PATH = PROJECT_ROOT / "synthetic_generator" / "synthetic_social_media_posts.csv"
KPI_PATH = PROCESSED_DIR / "kpi_dataset.csv"

## Load KPI and Aggregate Business Features

In [2]:
if KPI_PATH.exists():
    df = pd.read_csv(KPI_PATH, parse_dates=["post_date"])
else:
    df = engineer_kpis(clean_dataset(load_raw_dataset(RAW_DATA_PATH)))
df.head()

,business_name,sector,followers_count,post_date,posting_hour,day_of_week,month,post_type,caption_text,caption_length,...,is_reel,posting_time_bin,caption_length_bin,hashtags_count_bin,emoji_count_bin,engagement_level,business_size_bin,high_engagement,high_view_rate,high_comment_rate
0,LOFT Palestine,Fashion,4392.0,2026-03-31,0.0,Tuesday,3,reel,إطلالة أنثوية بلمسة عصرية ✨ من LOFT. للطلب الت...,88,...,1,night,medium,none,few,medium,small,0,0,0
1,LOFT Palestine,Fashion,4392.0,2026-03-18,0.0,Wednesday,3,reel,اختيارك المثالي لإطلالة كاجول مميزة 🤍✨ LOFT كو...,111,...,1,night,medium,none,few,high,small,1,1,0
2,LOFT Palestine,Fashion,4392.0,2026-03-16,0.0,Monday,3,reel,ستايل شبابي بلمسة عصرية له ولها ✨ اختيارات ممي...,133,...,1,night,medium,none,few,high,small,1,1,1
3,LOFT Palestine,Fashion,4392.0,2026-03-07,0.0,Saturday,3,reel,ستايل يناسبك، وتجربة تستحق الزيارة ✨ زورونا في...,82,...,1,night,medium,none,few,high,small,1,1,0
4,LOFT Palestine,Fashion,4392.0,2026-03-05,0.0,Thursday,3,reel,#LOFTSTYLE #Menstyle,21,...,1,night,short,few,none,high,small,1,1,0


In [3]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score

biz = aggregate_business_features(df)
features = ["followers_count","posts_count","engagement_mean","engagement_rate_mean","view_rate_mean","comment_rate_mean","promo_share","cta_share","location_share","dialect_share","discount_mean","video_share","reel_share"]
X = StandardScaler().fit_transform(biz[features].fillna(0))

## Clustering Experiments

In [4]:
rows, store = [], {}
for k in range(3,9):
    y = KMeans(n_clusters=k, random_state=42, n_init=20).fit_predict(X)
    rows.append({"model":"kmeans","setting":f"k={k}","silhouette":silhouette_score(X,y),"clusters":len(np.unique(y))})
    store[("kmeans",f"k={k}")] = y
for k in range(3,9):
    y = GaussianMixture(n_components=k, random_state=42).fit_predict(X)
    rows.append({"model":"gmm","setting":f"components={k}","silhouette":silhouette_score(X,y),"clusters":len(np.unique(y))})
    store[("gmm",f"components={k}")] = y
for k in range(3,9):
    for link in ["ward","average","complete"]:
        y = AgglomerativeClustering(n_clusters=k, linkage=link).fit_predict(X)
        rows.append({"model":"hierarchical","setting":f"k={k},linkage={link}","silhouette":silhouette_score(X,y),"clusters":len(np.unique(y))})
        store[("hierarchical",f"k={k},linkage={link}")] = y

exp = rank_models(pd.DataFrame(rows), higher_is_better_cols=["silhouette","clusters"])
best = exp.iloc[0]
biz["cluster_id"] = store[(best["model"], best["setting"])]
profiles = biz.groupby("cluster_id", as_index=False).agg(
    businesses=("business_name","size"),
    followers_count=("followers_count","mean"),
    engagement_rate_mean=("engagement_rate_mean","mean"),
    view_rate_mean=("view_rate_mean","mean"),
    comment_rate_mean=("comment_rate_mean","mean"),
    promo_share=("promo_share","mean"),
    discount_mean=("discount_mean","mean"),
).sort_values("engagement_rate_mean", ascending=False).reset_index(drop=True)
segments = ["community storytellers","reach-focused brands","discount-driven sellers","passive businesses","conversion-tactical players","video-first builders","niche loyalty brands","emerging challengers"]
profiles["cluster_label"] = [segments[i] if i < len(segments) else f"segment_{i}" for i in range(len(profiles))]
biz = biz.merge(profiles[["cluster_id","cluster_label"]], on="cluster_id", how="left")

## Save Outputs

In [5]:
biz.to_csv(PROCESSED_DIR / "business_clusters.csv", index=False)
profiles.to_csv(PROCESSED_DIR / "business_cluster_profiles.csv", index=False)
exp.to_csv(REPORTS_DIR / "business_clustering_experiments.csv", index=False)
display(exp.head(15))
display(profiles)
print("Insight: business segments enable targeted SME coaching and campaign allocation.")

,model,setting,silhouette,clusters,composite_score
0,hierarchical,"k=8,linkage=average",0.373312,8,1.553822
1,hierarchical,"k=7,linkage=average",0.398737,7,1.446288
2,hierarchical,"k=7,linkage=complete",0.398737,7,1.446288
3,hierarchical,"k=8,linkage=ward",0.336781,8,1.420967
4,kmeans,k=8,0.325964,8,1.381627
5,kmeans,k=7,0.332474,7,1.205301
6,hierarchical,"k=6,linkage=average",0.379150,6,1.175054
7,hierarchical,"k=6,linkage=complete",0.379150,6,1.175054
8,gmm,components=8,0.264728,8,1.158924
9,hierarchical,"k=8,linkage=complete",0.264513,8,1.158143


,cluster_id,businesses,followers_count,engagement_rate_mean,view_rate_mean,comment_rate_mean,promo_share,discount_mean,cluster_label
0,4,1,2.362000e+03,0.547126,28.157070,0.008256,0.000000,0.000000,community storytellers
1,1,2,3.400000e+06,0.046208,0.890024,0.000920,0.500000,0.000000,reach-focused brands
2,0,2,5.446000e+03,0.027441,78.523917,0.000347,0.462500,2.500000,discount-driven sellers
3,7,1,2.900000e+03,0.024713,0.000000,0.005172,0.000000,0.000000,passive businesses
4,3,1,5.700000e+06,0.020731,0.807289,0.000538,0.354839,3.064516,conversion-tactical players
5,2,53,2.252672e+05,0.005149,0.705607,0.000194,0.781351,1.451013,video-first builders
6,5,1,1.100000e+06,0.002245,0.000000,0.000053,1.000000,100.000000,niche loyalty brands
7,6,4,1.500000e+07,0.001415,0.064396,0.000030,0.012500,0.000000,emerging challengers


Insight: business segments enable targeted SME coaching and campaign allocation.
